# Music Annotation Project — Notebook 38
## V8.2i Envelope Relaxation and Perceptual Severity Calibration

**Status:** Development-only, **single-reviewer exploratory**. CPU runtime is sufficient. Run cells in order.

This notebook reuses frozen Notebook 37 outputs to test whether D3–D8 numerical alerts align with perceptual harm. It does not retrain, refit, recalibrate, reinterpret truth, adopt a lower threshold, authorise production use, or consume the untouched future validation cohort.

It preserves registered D3–D8 and truth-defined catastrophic flags, constructs source-group-separated calibration/audit strata, provides a blinded whole-page-first Gradio review, locks first-pass judgements before diagnostics, analyses the predeclared relaxation families, and creates a hashed upload bundle.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" pandas numpy pyarrow matplotlib pillow scipy


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, itertools, json, math, os, shutil, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB37_ROOT=ROOT/'observed_print_classifier_v8_2h_threshold_frontier'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2i_envelope_relaxation'
SOURCE_NB37_RUN_OVERRIDE=None  # Prefer the exact completed Notebook 37 run path.

NOTEBOOK_VERSION='38-v8.2i-envelope-relaxation'
PROTOCOL_VERSION='1.0-single-reviewer-exploratory'
REVIEWER_ID='reviewer_01'; REVIEWER_COUNT=1
ASSIGNMENT_SEED=38001; BOOTSTRAP_SEED=37001; BOOTSTRAP_RESAMPLES=10_000
CALIBRATION_GROUP_FRACTION=0.50
THRESHOLDS=[.995,.990,.985,.980,.975,.950,.925,.900,.875,.850,.825,.800,.750,.700,.600,.500]
D4_POINT=[.10,.15,.20,.25,.35,.50]; D4_UPPER=[.20,.25,.30,.40,.50,.75]
D5_POINT=[.005,.01,.02,.05,.10]; D5_UPPER=[.01,.02,.05,.10,.15]
FP_WEIGHTS=[1,2,3,5,10]; P1_WEIGHTS=[0,1]; P2_WEIGHTS=[5,10,25]; P3_WEIGHTS=[100,250,500]
D7_TP=[100,250,500,750,1000]; D7_PAGES=[5,8,10,15]; D7_GROUPS=[2,3,4,5]

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2i_envelope_relaxation'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; REVIEW=OUT/'review'; AUDIT=OUT/'audit'; VISUALS=OUT/'visuals'; REF=OUT/'reference'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,VISUALS,REF]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Helpers and pre-review protocol lock


In [ ]:
def now(): return datetime.now(timezone.utc).isoformat()
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def write_json(path,obj):
    path=Path(path); tmp=path.with_name(path.name+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str)); os.replace(tmp,path)
def write_csv(df,path):
    path=Path(path); tmp=path.with_name(path.name+'.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,path)
def fail(stage,msg,details=None):
    write_json(AUDIT/f'failure_{stage}.json',{'stage':stage,'message':msg,'details':details or {},'utc':now()})
    raise RuntimeError(f'{stage}: {msg}')
def col(df,names,required=True):
    d={str(x).lower():x for x in df.columns}
    for n in names:
        if n.lower() in d:return d[n.lower()]
    if required: raise KeyError(f'Missing one of {names}; columns={list(df.columns)}')
def atom_record(d):
    return hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest()
def valid_image(p):
    try:
        with Image.open(p) as im: im.verify()
        return True
    except: return False

def protocol():
    return {
      'notebook_version':NOTEBOOK_VERSION,'protocol_version':PROTOCOL_VERSION,'created_utc':now(),
      'status':'development-only-single-reviewer-exploratory','reviewer_count':1,'reviewer_id':REVIEWER_ID,
      'may_nominate_revised_envelope_for_independent_test':False,'production_authority':False,
      'untouched_future_validation_used':False,'thresholds':THRESHOLDS,'assignment_seed':ASSIGNMENT_SEED,
      'bootstrap_seed':BOOTSTRAP_SEED,'bootstrap_resamples':BOOTSTRAP_RESAMPLES,
      'calibration_group_fraction':CALIBRATION_GROUP_FRACTION,'registered_utility':'TP-5*FP-500*C',
      'D4_point_ceilings':D4_POINT,'D4_upper_ceilings':D4_UPPER,'D5_point_ceilings':D5_POINT,'D5_upper_ceilings':D5_UPPER,
      'D6_weights':{'FP':FP_WEIGHTS,'P0':[0],'P1':P1_WEIGHTS,'P2':P2_WEIGHTS,'P3':P3_WEIGHTS,'P4':'hard_fail'},
      'D7_values':{'true_print':D7_TP,'pages':D7_PAGES,'source_groups':D7_GROUPS,'benefit':['B1+','B2+','B3+']},
      'sparse_hard_negative':{'role':'explicitly sparse training-role hard negatives','D9':'not validated without separately scored held-out sparse evidence'},
      'hard_stops':['D1 failure','D2 failure','P4','repeated P3 mechanism','truth exposed before first-pass lock','incomplete review']}
LOCK=protocol(); write_json(OUT/'protocol_lock.json',LOCK)
print(json.dumps(LOCK,indent=2))


## 3. Resolve and verify Notebook 37 inputs


In [ ]:
REQ=[
'experiment_identity.json','source_v82_record.json','executive_summary.txt',
'tables/threshold_frontier_summary.csv','tables/threshold_frontier_by_page.csv','tables/threshold_frontier_by_source_group.csv',
'tables/incremental_threshold_band_summary.csv','tables/incremental_threshold_pixels.parquet',
'tables/incremental_component_review_queue_all.csv','tables/catastrophic_components_all_thresholds.csv',
'tables/catastrophic_component_first_appearance_inventory.csv','tables/computable_gate_map.csv',
'tables/representative_page_selection.csv','audit/whole_page_integrity_and_hashes.csv','upload/upload_manifest.csv']

def runs(root):
    if not Path(root).exists(): return []
    return sorted([p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in REQ[:8])],key=lambda p:p.stat().st_mtime)
NB37=Path(SOURCE_NB37_RUN_OVERRIDE) if SOURCE_NB37_RUN_OVERRIDE else (runs(NB37_ROOT)[-1] if runs(NB37_ROOT) else None)
if NB37 is None: fail('input','No complete-looking Notebook 37 run found',{'root':str(NB37_ROOT)})
rows=[]
for r in REQ:
    p=NB37/r; rows.append({'relative_path':r,'exists':p.exists(),'bytes':p.stat().st_size if p.is_file() else np.nan,'sha256':sha(p) if p.is_file() else ''})
ia=pd.DataFrame(rows); write_csv(ia,AUDIT/'input_resolution_audit.csv')
if not ia.exists.all(): fail('input','Missing mandatory Notebook 37 artefacts',{'missing':ia.loc[~ia.exists,'relative_path'].tolist()})

# Verify source upload manifest where every listed file must resolve and match any supplied SHA-256.
um=pd.read_csv(NB37/'upload/upload_manifest.csv'); pc=col(um,['relative_path','path','file','filename']); hc=col(um,['sha256','sha_256','hash'],False)
vr=[]
for _,r in um.iterrows():
    rel=str(r[pc]); cand=[NB37/rel,NB37/'upload'/rel]; p=next((x for x in cand if x.is_file()),None)
    actual=sha(p) if p else ''; expected=str(r[hc]) if hc and pd.notna(r[hc]) else ''
    vr.append({'manifest_path':rel,'resolved':str(p or ''),'exists':bool(p),'expected_sha256':expected,'actual_sha256':actual,'hash_matches':actual.lower()==expected.lower() if expected and p else np.nan})
vr=pd.DataFrame(vr); write_csv(vr,AUDIT/'nb37_manifest_verification.csv')
if (~vr.exists).any() or vr.hash_matches.eq(False).any(): fail('manifest','Notebook 37 upload manifest did not verify')

frontier=pd.read_csv(NB37/'tables/threshold_frontier_summary.csv')
actual=[round(float(x),3) for x in frontier[col(frontier,['threshold'])]]
if actual!=THRESHOLDS: fail('thresholds','Threshold identity/order mismatch',{'expected':THRESHOLDS,'actual':actual})
LOCK.update({'source_nb37_run':str(NB37),'source_identity_sha256':sha(NB37/'experiment_identity.json'),'source_v82_record_sha256':sha(NB37/'source_v82_record.json'),'source_selection':'override' if SOURCE_NB37_RUN_OVERRIDE else 'latest_complete_looking'})
write_json(OUT/'protocol_lock.json',LOCK)
shutil.copy2(NB37/'experiment_identity.json',REF/'notebook37_experiment_identity.json'); shutil.copy2(NB37/'source_v82_record.json',REF/'source_v82_record.json')
print('Verified source:',NB37)


## 4. Registered D3–D8 replay


In [ ]:
needed=['threshold','true_print','false_annotation','truth_uncertain_print','selected_print_pixels','utility','catastrophic_components','false_true_ratio','false_preservation_rate','wilson95_upper_false_preservation','bootstrap_utility_95_low','bootstrap_false_true_95_high','benefiting_pages','benefiting_source_groups','source_stability_for_groups_with_3_pages','D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability','core_development_envelope_D3_to_D8']
miss=[x for x in needed if x not in frontier]
if miss: fail('gate_replay','Missing registered fields',{'columns':miss})
replay=pd.DataFrame({'threshold':frontier.threshold,'D3':frontier.catastrophic_components.eq(0),'D4':frontier.false_true_ratio.le(.10)&frontier.bootstrap_false_true_95_high.le(.20),'D5':frontier.false_preservation_rate.le(.005)&frontier.wilson95_upper_false_preservation.le(.01),'D6':frontier.utility.gt(0)&frontier.bootstrap_utility_95_low.gt(0),'D7':frontier.true_print.ge(500)&frontier.benefiting_pages.ge(10)&frontier.benefiting_source_groups.ge(3),'D8':frontier.source_stability_for_groups_with_3_pages.astype(bool)})
source_cols=['D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability']
for i,s in enumerate(source_cols,3):
    if not np.array_equal(replay[f'D{i}'].to_numpy(),frontier[s].astype(bool).to_numpy()): fail('gate_replay',f'D{i} mismatch')
write_csv(frontier[needed],TABLES/'registered_D3_D8_results.csv'); write_csv(replay,AUDIT/'registered_gate_replay.csv')
print('Registered D3–D8 replay verified.')


## 5. Stable review units and source-group split


In [ ]:
q=pd.read_csv(NB37/'tables/incremental_component_review_queue_all.csv'); cf=pd.read_csv(NB37/'tables/catastrophic_component_first_appearance_inventory.csv')
alias={
'page':['page'],'source_group':['source_group'],'mask_component_id':['mask_component_id','component_id'],'higher_threshold':['higher_threshold','from_threshold','high_threshold'],'lower_threshold':['lower_threshold','to_threshold','low_threshold'],
'new_selected':['new_selected','new_selected_pixels','incremental_selected','selected'],'new_true_print':['new_true_print','incremental_true_print','true_print'],'new_false_annotation':['new_false_annotation','incremental_false_annotation','false_annotation'],'new_truth_uncertain':['new_truth_uncertain','incremental_truth_uncertain','truth_uncertain'],
'local_panel':['local_panel','local_crop','local_path','local_crop_path'],'wide_panel':['wide_panel','wide_crop','wide_path','wide_crop_path'],'locator':['locator','whole_page_locator','locator_path','whole_page_locator_path'],'truth_overlay':['truth_overlay','cumulative_truth_overlay','incremental_truth_overlay','truth_overlay_path']}
u=pd.DataFrame()
for target,names in alias.items():
    c=col(q,names,target in ['page','source_group','mask_component_id','higher_threshold','lower_threshold'])
    if c:u[target]=q[c]
for c in ['new_selected','new_true_print','new_false_annotation','new_truth_uncertain']:
    if c not in u:u[c]=0
    u[c]=pd.to_numeric(u[c],errors='coerce').fillna(0).astype(int)
u.page=u.page.astype(str); u.source_group=u.source_group.astype(str); u.mask_component_id=pd.to_numeric(u.mask_component_id).astype(int)
u.higher_threshold=pd.to_numeric(u.higher_threshold).round(3); u.lower_threshold=pd.to_numeric(u.lower_threshold).round(3)
cp=col(cf,['page']); cc=col(cf,['mask_component_id','component_id']); ct=col(cf,['first_appearance_threshold','threshold','lower_threshold'])
c=cf[[cp,cc,ct]].copy(); c.columns=['page','mask_component_id','first_appearance_threshold']; c.page=c.page.astype(str); c.mask_component_id=pd.to_numeric(c.mask_component_id).astype(int); c['registered_catastrophic']=True
u=u.merge(c,on=['page','mask_component_id'],how='left'); u.registered_catastrophic=u.registered_catastrophic.fillna(False).astype(bool)
u['lineage_key']=u.apply(lambda r:f'{r.page}|{r.source_group}|{r.mask_component_id}|{r.higher_threshold:.3f}|{r.lower_threshold:.3f}',axis=1)
u['review_unit_id']=u.lineage_key.map(lambda x:'RU_'+hashlib.sha256(f'{ASSIGNMENT_SEED}|{x}'.encode()).hexdigest()[:16])
if u.review_unit_id.duplicated().any(): fail('units','Duplicate review units')
def asset(v):
    if pd.isna(v) or str(v).strip()=='':return ''
    p=Path(str(v))
    for z in [p,NB37/p,NB37/'review'/p,NB37/'visual_audit'/p]:
        if z.is_file():return str(z)
    return ''
for x in ['local_panel','wide_panel','locator','truth_overlay']:
    if x not in u:u[x]=''
    else:u[x]=u[x].map(asset)

# Deterministic whole-source-group split, balancing large/catastrophic groups first.
g=u.groupby('source_group',as_index=False).agg(review_units=('review_unit_id','size'),catastrophic_units=('registered_catastrophic','sum'),pages=('page','nunique'))
rng=np.random.default_rng(ASSIGNMENT_SEED); g['_tie']=rng.random(len(g)); g=g.sort_values(['catastrophic_units','review_units','_tie'],ascending=[False,False,True])
target=round(len(u)*CALIBRATION_GROUP_FRACTION); ca=[]; au=[]; n=0
for _,r in g.iterrows():
    (ca if n<target else au).append(r.source_group)
    if n<target:n+=int(r.review_units)
if len(g)>=2 and (not ca or not au):ca=[g.iloc[0].source_group]; au=g.iloc[1:].source_group.tolist()
g['assignment_split']=np.where(g.source_group.isin(ca),'rubric_calibration','audit_reserve'); g=g.drop(columns='_tie')
u=u.merge(g[['source_group','assignment_split']],on='source_group',validate='many_to_one')
write_csv(g,TABLES/'review_split_map.csv'); write_csv(u,TABLES/'review_unit_manifest.csv')
print('Units:',len(u),'groups:',u.source_group.nunique(),'calibration groups:',len(ca),'audit groups:',len(au))


## 6. Blinded A/B material


In [ ]:
def tokens(t):return [f'{t:.3f}',f'{t:.3f}'.replace('.',''),f'{t:.3f}'.replace('.','_'),str(float(t))]
def whole(page,t):
    d=NB37/'whole_pages'/str(page)
    if not d.exists():return None
    scores=[]
    for p in d.rglob('*'):
        if p.suffix.lower() not in ['.png','.jpg','.jpeg']:continue
        s=sum(x in p.name.lower() for x in tokens(t))-3*sum(x in p.name.lower() for x in ['difference','overlay','decision','locator'])
        if float(t)==.995 and ('registered' in p.name.lower() or 'v8_2' in p.name.lower()):s+=2
        if s>0:scores.append((s,p))
    return max(scores,key=lambda z:z[0])[1] if scores else None
def observed(page):
    d=NB37/'whole_pages'/str(page)
    for pat in ['observed.png','*observed*.png','*annotated*.png']:
        x=list(d.glob(pat)) if d.exists() else []
        if x:return str(x[0])
    return ''
rows=[]; missing=[]
for _,r in u.iterrows():
    hi=whole(r.page,r.higher_threshold); lo=whole(r.page,r.lower_threshold)
    if not hi or not lo or not valid_image(hi) or not valid_image(lo):
        missing.append({'review_unit_id':r.review_unit_id,'page':r.page,'higher_path':str(hi or ''),'lower_path':str(lo or '')}); continue
    left_low=int(hashlib.sha256(f'{ASSIGNMENT_SEED}|{r.review_unit_id}'.encode()).hexdigest(),16)%2==0
    A,B=(lo,hi) if left_low else (hi,lo); mapping={'A':'lower','B':'higher'} if left_low else {'A':'higher','B':'lower'}
    rows.append({**r.to_dict(),'whole_page_A':str(A),'whole_page_B':str(B),'A_sha256':sha(A),'B_sha256':sha(B),'observed_path':observed(r.page),'truth_overlay_path':r.truth_overlay,'randomisation_side_locked':base64.urlsafe_b64encode(json.dumps(mapping).encode()).decode()})
write_csv(pd.DataFrame(missing),AUDIT/'missing_review_assets.csv')
if missing:fail('review_assets','Unreadable/missing whole-page pairs',{'count':len(missing)})
blind=pd.DataFrame(rows); write_csv(blind,TABLES/'blinded_review_manifest.csv')
print('Blinded units:',len(blind))


## 7. Gradio review UI

The first pass shows only randomised whole pages. Threshold, truth, source group and comparator identity are hidden. Saving locks the first-pass record. Diagnostics then reveal any available wide/local/locator images and observed input; diagnostic ratings are stored separately and never overwrite round one.


In [ ]:
import gradio as gr
R1=REVIEW/'perceptual_review_round1.csv'; R2=REVIEW/'diagnostic_review_round2.csv'
R1C=['review_unit_id','reviewer_id','initial_visibility','pair_preference','harm_severity','benefit_severity','defect_scope','musical_function_at_risk','annotation_prominence','reviewer_action','confidence','review_note','timestamp_utc','locked_before_diagnostic','record_sha256']
R2C=['review_unit_id','reviewer_id','diagnostic_mechanism','resolved_harm_severity','resolved_benefit_severity','diagnostic_note','timestamp_utc','record_sha256']
def records(path,cols):
    if not Path(path).exists():return pd.DataFrame(columns=cols)
    d=pd.read_csv(path,keep_default_na=False)
    for c in cols:
        if c not in d:d[c]=''
    return d[cols]
def save(path,cols,d):
    d['record_sha256']=atom_record(d); z=records(path,cols); z=z[z.review_unit_id.astype(str)!=str(d['review_unit_id'])]; write_csv(pd.concat([z,pd.DataFrame([d])],ignore_index=True)[cols],path); return d['record_sha256']
def pending():
    done=set(records(R2,R2C).review_unit_id.astype(str)); return [i for i,x in enumerate(blind.review_unit_id.astype(str)) if x not in done]
def image(x):return str(x) if str(x) and Path(str(x)).is_file() else None
def view(i):
    i=int(max(0,min(i,len(blind)-1)));r=blind.iloc[i];n=len(records(R2,R2C));return i,image(r.whole_page_A),image(r.whole_page_B),f'Item {i+1}/{len(blind)} — complete {n}/{len(blind)} — single-reviewer exploratory',r.review_unit_id
def first(i,vis,pref,harm,benefit,scope,musical,prom,action,conf,note):
    r=blind.iloc[int(i)]; vals=[vis,pref,harm,benefit,scope,musical,prom,action,conf]
    if any(v in [None,''] for v in vals):raise gr.Error('Complete all first-pass fields.')
    if (harm in ['P2','P3','P4','U'] or benefit in ['B3','B4','U']) and not str(note).strip():raise gr.Error('Note required for P2–P4, B3–B4 or U.')
    d=dict(zip(R1C[:-3], [r.review_unit_id,REVIEWER_ID,vis,pref,harm,benefit,scope,musical,prom,action,conf,str(note).strip()]))
    d.update({'timestamp_utc':now(),'locked_before_diagnostic':True}); h=save(R1,R1C,d)
    return f'First pass locked — {h[:12]}',gr.update(visible=False),gr.update(visible=True),image(r.wide_panel),image(r.local_panel),image(r.locator),image(r.observed_path),image(r.truth_overlay_path)
def second(i,mechanism,harm,benefit,note):
    r=blind.iloc[int(i)]
    if not mechanism or not harm or not benefit:raise gr.Error('Complete diagnostic fields.')
    d={'review_unit_id':r.review_unit_id,'reviewer_id':REVIEWER_ID,'diagnostic_mechanism':mechanism,'resolved_harm_severity':harm,'resolved_benefit_severity':benefit,'diagnostic_note':str(note).strip(),'timestamp_utc':now()};h=save(R2,R2C,d)
    p=pending(); nxt=next((x for x in p if x>int(i)),p[0] if p else int(i));v=view(nxt)
    return *v,f'Diagnostic saved — {h[:12]}',gr.update(visible=True),gr.update(visible=False)

with gr.Blocks(title='Notebook 38 Review') as app:
    gr.Markdown('# Notebook 38 — blinded perceptual review')
    state=gr.State(pending()[0] if pending() else 0); unit=gr.Textbox(label='Opaque unit',interactive=False); status=gr.Markdown()
    with gr.Row():A=gr.Image(label='A',type='filepath',height=700);B=gr.Image(label='B',type='filepath',height=700)
    with gr.Column(visible=True) as p1:
        vis=gr.Radio(['no difference','visible difference','unclear'],label='Initial visibility'); pref=gr.Radio(['A','B','neither','mixed','unclear'],label='Preference')
        with gr.Row(): hs=gr.Radio(['P0','P1','P2','P3','P4','U'],label='Harm');bs=gr.Radio(['B0','B1','B2','B3','B4','U'],label='Benefit')
        scope=gr.Dropdown(['pixel-fragment','component-local','staff/system','page-repeated','page-global'],label='Defect scope'); musical=gr.Dropdown(['none','staff','notehead','stem/beam','accidental','rest','articulation','dynamic','text/lyric','barline/repeat','other','unclear'],label='Musical function at risk')
        prom=gr.Radio(['none','faint','moderate','prominent'],label='Annotation prominence'); action=gr.Radio(['accept','accept-with-note','manual-correct','reject-policy-case','unclear'],label='Action'); conf=gr.Radio(['low','medium','high'],label='Confidence'); note1=gr.Textbox(label='First-pass note',lines=3); b1=gr.Button('Lock first pass and reveal diagnostics',variant='primary')
    with gr.Column(visible=False) as p2:
        gr.Markdown('## Diagnostics — first pass locked')
        with gr.Row():wide=gr.Image(label='Wide');local=gr.Image(label='Local');locator=gr.Image(label='Locator');obs=gr.Image(label='Observed');truth=gr.Image(label='Truth/overlay diagnostic')
        mech=gr.Dropdown(['correct print recovery','retained annotation','mixed overlap','registration ambiguity','truth-boundary sensitivity','display artefact','other'],label='Mechanism')
        with gr.Row():rh=gr.Radio(['P0','P1','P2','P3','P4','U'],label='Resolved harm');rb=gr.Radio(['B0','B1','B2','B3','B4','U'],label='Resolved benefit')
        note2=gr.Textbox(label='Diagnostic note',lines=3);b2=gr.Button('Save and next',variant='primary')
    msg=gr.Markdown();app.load(view,[state],[state,A,B,status,unit]);b1.click(first,[state,vis,pref,hs,bs,scope,musical,prom,action,conf,note1],[msg,p1,p2,wide,local,locator,obs,truth]);b2.click(second,[state,mech,rh,rb,note2],[state,A,B,status,unit,msg,p1,p2])
app.launch(share=True,debug=False)


## 8. Review completion and severity-map lock


In [ ]:
r1=records(R1,R1C);r2=records(R2,R2C);expected=set(blind.review_unit_id.astype(str));a=set(r1.review_unit_id.astype(str));b=set(r2.review_unit_id.astype(str))
coverage=pd.DataFrame({'review_unit_id':sorted(expected),'round1_complete':[x in a for x in sorted(expected)],'round2_complete':[x in b for x in sorted(expected)]});write_csv(coverage,AUDIT/'review_coverage.csv')
if not coverage[['round1_complete','round2_complete']].all().all():fail('review','Review incomplete',{'expected':len(expected),'round1':len(a&expected),'round2':len(b&expected)})
if not r1.locked_before_diagnostic.astype(str).str.lower().isin(['true','1']).all():fail('review','First-pass lock violation')
write_csv(pd.DataFrame([{'reviewer_count':1,'raw_agreement':np.nan,'exact_agreement':np.nan,'within_one_level_agreement':np.nan,'status':'not_estimable_single_reviewer'}]),TABLES/'reviewer_agreement.csv')
joined=blind.merge(r1,on='review_unit_id',validate='one_to_one').merge(r2,on=['review_unit_id','reviewer_id'],validate='one_to_one')
cal=joined[joined.assignment_split.eq('rubric_calibration')]; audit_reviews=joined[joined.assignment_split.eq('audit_reserve')]
if cal.empty or audit_reviews.empty:fail('split','Empty calibration or audit reserve')
sev={'protocol_version':PROTOCOL_VERSION,'frozen_utc':now(),'calibration_source_groups':sorted(cal.source_group.unique()),'calibration_units':len(cal),'calibration_manifest_sha256':hashlib.sha256(cal[['review_unit_id','resolved_harm_severity','resolved_benefit_severity']].to_csv(index=False).encode()).hexdigest(),'weights':LOCK['D6_weights'],'mapping_tuned_to_audit_reserve':False,'single_reviewer_exploratory':True}
write_json(OUT/'severity_mapping_lock.json',sev);print(json.dumps(sev,indent=2))


## 9. Audit-reserve D3–D8 sensitivity analysis


In [ ]:
hn={'P0':0,'P1':1,'P2':2,'P3':3,'P4':4};bn={'B0':0,'B1':1,'B2':2,'B3':3,'B4':4}
audit_reviews['harm_n']=audit_reviews.resolved_harm_severity.map(hn);audit_reviews['benefit_n']=audit_reviews.resolved_benefit_severity.map(bn)
if audit_reviews[['harm_n','benefit_n']].isna().any().any():fail('analysis','Unresolved U remains in audit reserve')
by_page=pd.read_csv(NB37/'tables/threshold_frontier_by_page.csv');by_source=pd.read_csv(NB37/'tables/threshold_frontier_by_source_group.csv')
audit_groups=set(g.loc[g.assignment_split.eq('audit_reserve'),'source_group'].astype(str));pg=by_page[by_page.source_group.astype(str).isin(audit_groups)];sg=by_source[by_source.source_group.astype(str).isin(audit_groups)]
def active(t):return audit_reviews[audit_reviews.lower_threshold.astype(float).ge(t)]
def wilson(e,n,z=1.96):
    if n<=0:return np.nan
    p=e/n;d=1+z*z/n;return (p+z*z/(2*n)+z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)))/d
def boot(pages,seed):
    v=pages[['true_print','false_annotation','truth_uncertain_print','catastrophic_components']].to_numpy(float);rng=np.random.default_rng(seed);out=[]
    for _ in range(0,BOOTSTRAP_RESAMPLES,1000):
        x=v[rng.integers(0,len(v),size=(min(1000,BOOTSTRAP_RESAMPLES-len(out)*1000),len(v)))].sum(1);tp,fp,tu,c=x.T;out.append(np.c_[tp-5*fp-500*c,fp/np.maximum(tp,1),fp/np.maximum(tp+fp+tu,1)])
    return np.vstack(out)
P=[];U=[];D7=[];D8=[]
for t in THRESHOLDS:
    p=pg[np.isclose(pg.threshold,t)];s=sg[np.isclose(sg.threshold,t)];r=active(t)
    if p.empty:continue
    tp=int(p.true_print.sum());fp=int(p.false_annotation.sum());tu=int(p.truth_uncertain_print.sum());cats=int(p.catastrophic_components.sum());selected=int(p.selected.sum());bt=boot(p,BOOTSTRAP_SEED+round(1000*t))
    p2=r.harm_n.ge(2);p3=r.harm_n.ge(3);p4=r.harm_n.eq(4);rep3=bool((r.loc[p3].groupby(['source_group','diagnostic_mechanism']).size()>=2).any()) if p3.any() else False
    row={'threshold':t,'true_print':tp,'false_annotation':fp,'truth_uncertain_print':tu,'selected':selected,'catastrophes':cats,'utility':tp-5*fp-500*cats,'utility_95_low':np.quantile(bt[:,0],.025),'false_true':fp/max(tp,1),'false_true_95_high':np.quantile(bt[:,1],.975),'false_preservation':fp/max(selected,1),'false_preservation_wilson_upper':wilson(fp,selected),'P2_plus':int(p2.sum()),'P3_plus':int(p3.sum()),'P4':int(p4.sum()),'repeated_P3':rep3,'D3_R0':cats==0,'D3_R1':bool((~r.registered_catastrophic|r.harm_n.le(1)).all() and not p2.any()),'D3_R2':bool(not p3.any() and not ((r.loc[p2].groupby(['source_group','diagnostic_mechanism']).size()>=2).any() if p2.any() else False)),'D3_R3':not p3.any()}
    P.append(row)
    counts=r.resolved_harm_severity.value_counts().to_dict()
    for w,a,b,c in itertools.product(FP_WEIGHTS,P1_WEIGHTS,P2_WEIGHTS,P3_WEIGHTS):
        penalty=counts.get('P1',0)*a+counts.get('P2',0)*b+counts.get('P3',0)*c;U.append({'threshold':t,'FP_weight':w,'P1_weight':a,'P2_weight':b,'P3_weight':c,'P4_hard_fail':bool(p4.any()),'severity_adjusted_utility':tp-w*fp-penalty})
    for label,minb in [('B1+',1),('B2+',2),('B3+',3)]:
        z=r[r.harm_n.le(1)&r.benefit_n.ge(minb)];ztp=int(z.new_true_print.sum());zp=z.loc[z.new_true_print.gt(0),'page'].nunique();zg=z.loc[z.new_true_print.gt(0),'source_group'].nunique()
        for x,y,v in itertools.product(D7_TP,D7_PAGES,D7_GROUPS):D7.append({'threshold':t,'benefit':label,'qualified_tp':ztp,'qualified_pages':zp,'qualified_groups':zg,'required_tp':x,'required_pages':y,'required_groups':v,'passes':ztp>=x and zp>=y and zg>=v and not p4.any()})
    for _,x in s.iterrows():
        rr=r[r.source_group.astype(str).eq(str(x.source_group))];q3=int(rr.harm_n.ge(3).sum());q4=int(rr.harm_n.eq(4).sum());repeat=bool((rr.loc[rr.harm_n.eq(3)].groupby('diagnostic_mechanism').size()>=2).any()) if q3 else False
        D8.append({'threshold':t,'source_group':x.source_group,'pages':int(x.pages),'false_true_ratio':float(x.false_true_ratio),'catastrophes':int(x.catastrophic_components),'utility':float(x.utility),'P3':q3,'P4':q4,'D8_R0':int(x.pages)<3 or (float(x.false_true_ratio)<=.25 and int(x.catastrophic_components)==0),'D8_R1':float(x.false_true_ratio)<=.35 and q3==0 and float(x.utility)>=0,'D8_R2':float(x.false_true_ratio)<=.50 and q4==0 and not repeat,'D8_R3_source_scoped_only':True})
perceptual=pd.DataFrame(P);util=pd.DataFrame(U);d7=pd.DataFrame(D7);d8=pd.DataFrame(D8)
for df,name in [(perceptual,'perceptual_D3_D8_results.csv'),(perceptual,'audit_reserve_envelope_results.csv'),(util,'severity_adjusted_utility_matrix.csv'),(d7,'D7_sensitivity_matrix.csv'),(d8,'D8_source_stability_variants.csv'),(pg,'results_by_page.csv'),(sg,'results_by_source_group.csv')]:write_csv(df,TABLES/name)
print(perceptual.to_string(index=False))


## 10. Influence, frontier, verdict and visual outputs


In [ ]:
burden=[];logo=[];front=[]
for t in THRESHOLDS:
    r=active(t);p=pg[np.isclose(pg.threshold,t)]
    if len(r):burden.append({'threshold':t,'units':len(r),'P2_plus':int(r.harm_n.ge(2).sum()),'P3_plus':int(r.harm_n.ge(3).sum()),'P4':int(r.harm_n.eq(4).sum()),'review_burden_rate':float(r.harm_n.ge(2).mean())})
    for omit in sorted(audit_groups):
        z=p[~p.source_group.astype(str).eq(omit)]
        if len(z):
            tp=int(z.true_print.sum());fp=int(z.false_annotation.sum());c=int(z.catastrophic_components.sum());logo.append({'threshold':t,'omitted_source_group':omit,'true_print':tp,'false_annotation':fp,'catastrophes':c,'utility':tp-5*fp-500*c,'false_true':fp/max(tp,1)})
for _,r in perceptual.sort_values('threshold',ascending=False).iterrows():
    z=d8[np.isclose(d8.threshold,r.threshold)];front.append({'threshold':r.threshold,'true_print':r.true_print,'false_annotation':r.false_annotation,'P2_plus':r.P2_plus,'P3_plus':r.P3_plus,'P4':r.P4,'D3_R1':r.D3_R1,'D3_R2':r.D3_R2,'D3_R3':r.D3_R3,'D8_R1_all':bool(z.D8_R1.all()),'D8_R2_all':bool(z.D8_R2.all()),'hard_stop':bool(r.P4>0 or r.repeated_P3),'exploratory_only':True})
burden=pd.DataFrame(burden);logo=pd.DataFrame(logo);relax=pd.DataFrame(front);write_csv(burden,TABLES/'review_burden_summary.csv');write_csv(logo,TABLES/'leave_one_group_out.csv');write_csv(relax,TABLES/'envelope_relaxation_frontier.csv')

plt.figure(figsize=(8,5));plt.plot(perceptual.false_annotation,perceptual.true_print,marker='o')
for _,r in perceptual.iterrows():plt.annotate(f'{r.threshold:.3f}',(r.false_annotation,r.true_print),fontsize=8)
plt.xlabel('False-annotation pixels');plt.ylabel('True-print pixels');plt.title('Audit-reserve benefit versus truth error');plt.tight_layout();plt.savefig(VISUALS/'perceptual_frontier.png',dpi=160);plt.close()
sev=[]
for t in THRESHOLDS:
    for s,n in active(t).resolved_harm_severity.value_counts().items():sev.append({'threshold':t,'severity':s,'count':n})
if sev:
    pd.DataFrame(sev).pivot(index='threshold',columns='severity',values='count').fillna(0).sort_index(ascending=False).plot(kind='bar',stacked=True,figsize=(10,5));plt.tight_layout();plt.savefig(VISUALS/'severity_by_threshold.png',dpi=160);plt.close()

stop=relax[relax.hard_stop].head(1);first_stop=float(stop.iloc[0].threshold) if len(stop) else None
outcome='reject_relaxation_at_or_below_first_P4' if (perceptual.P4>0).any() else ('reject_general_relaxation_at_or_below_repeated_P3' if perceptual.repeated_P3.any() else ('potential_supervised_review_tier_exploratory' if (perceptual.P2_plus>0).any() else 'no_material_harm_observed_exploratory_only'))
verdict={'status':'single-reviewer-exploratory','outcome':outcome,'first_material_harm_or_instability_threshold':first_stop,'may_nominate_revised_envelope':False,'production_authority':False,'threshold_adoption_authority':False,'untouched_future_validation_used':False,'D9':'not_validated_training_role_sparse_material_only'};write_json(OUT/'notebook38_verdict.json',verdict)
summary='\n'.join(['Notebook 38 — V8.2i','Status: SINGLE-REVIEWER EXPLORATORY','Outcome: '+outcome,'First material-harm/instability threshold: '+str(first_stop),f'Review units: {len(joined)}; calibration: {len(cal)}; audit reserve: {len(audit_reviews)}','D9 sparse/hard-negative: not validated; existing material served a sparse training role.','Authority: no threshold adoption, independent-test nomination, or production use.'])+"\n";(OUT/'executive_summary.txt').write_text(summary)
(OUT/'decision_record.md').write_text('\n'.join(['# Notebook 38 Decision Record','', '- Status: single-reviewer exploratory.',f'- Outcome: `{outcome}`.',f'- First hard stop: `{first_stop}`.','- Registered D3–D8 retained.','- Agreement is not estimable with one reviewer.','- No threshold adoption or independent-test nomination.','']))
print(summary)


## 11. Curated final upload package


In [ ]:
items=[OUT/'protocol_lock.json',OUT/'severity_mapping_lock.json',OUT/'notebook38_verdict.json',OUT/'executive_summary.txt',OUT/'decision_record.md',TABLES/'review_unit_manifest.csv',TABLES/'review_split_map.csv',TABLES/'blinded_review_manifest.csv',R1,R2,TABLES/'reviewer_agreement.csv',TABLES/'registered_D3_D8_results.csv',TABLES/'perceptual_D3_D8_results.csv',TABLES/'audit_reserve_envelope_results.csv',TABLES/'envelope_relaxation_frontier.csv',TABLES/'results_by_page.csv',TABLES/'results_by_source_group.csv',TABLES/'leave_one_group_out.csv',TABLES/'review_burden_summary.csv',TABLES/'severity_adjusted_utility_matrix.csv',TABLES/'D7_sensitivity_matrix.csv',TABLES/'D8_source_stability_variants.csv',VISUALS/'perceptual_frontier.png',VISUALS/'severity_by_threshold.png',AUDIT/'input_resolution_audit.csv',AUDIT/'nb37_manifest_verification.csv',AUDIT/'registered_gate_replay.csv',AUDIT/'review_coverage.csv',AUDIT/'missing_review_assets.csv',REF/'notebook37_experiment_identity.json',REF/'source_v82_record.json']
missing=[str(x.relative_to(OUT)) for x in items if not x.exists()]
if missing:fail('package','Missing curated outputs',{'missing':missing})
UPLOAD.mkdir();rows=[]
for src in items:
    rel=src.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':rel.as_posix(),'bytes':dst.stat().st_size,'sha256':sha(dst)})
readme='Notebook 38 upload package. Single-reviewer exploratory: no threshold adoption, independent-test nomination, or production authority.\n';(UPLOAD/'README.txt').write_text(readme);rows.append({'relative_path':'README.txt','bytes':(UPLOAD/'README.txt').stat().st_size,'sha256':sha(UPLOAD/'README.txt')})
files=sorted(x.relative_to(UPLOAD).as_posix() for x in UPLOAD.rglob('*') if x.is_file());(UPLOAD/'upload_file_list.txt').write_text('\n'.join(files)+'\n');rows.append({'relative_path':'upload_file_list.txt','bytes':(UPLOAD/'upload_file_list.txt').stat().st_size,'sha256':sha(UPLOAD/'upload_file_list.txt')});write_csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv')
for _,r in pd.read_csv(UPLOAD/'upload_manifest.csv').iterrows():
    if sha(UPLOAD/r.relative_path)!=r.sha256:fail('package','Hash verification failed',{'file':r.relative_path})
zpath=OUT/'notebook38_upload_bundle.zip'
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file():z.write(p,arcname=(Path('upload')/p.relative_to(UPLOAD)).as_posix())
write_json(OUT/'final_package_record.json',{'zip':str(zpath),'bytes':zpath.stat().st_size,'sha256':sha(zpath),'utc':now()})
print('Final bundle:',zpath)


## Interpretation boundary

A completed run may describe perceptually negligible registered alerts, review burden, source sensitivity and the first material-harm transition. With one reviewer it remains exploratory and does not nominate a revised envelope. Any candidate must be separately frozen and tested on fresh untouched evidence.
